# 02 · Silver layer

Turns the bronze data into clean, labelled reviews ready for modelling: drop bad rows, remove duplicates, map ratings to sentiment labels, strip text that leaks the label, normalise the Arabic text, and split into train / val / test.

| | |
|---|---|
| **Input** | `data/bronze/reviews.parquet` |
| **Output** | `data/silver/hard_reviews/` (Parquet, partitioned by `split`) |

Spark settings and paths come from `src/config.py`, which also points Spark at the project's `winutils.exe` so it can write files on Windows.

In [1]:
from config import get_spark, BRONZE_PATH, SILVER_PATH, QUARANTINE_PATH   # also sets up winutils
from pyspark.sql import functions as F

spark = get_spark("silver")

In [2]:
bronze = spark.read.parquet("../data/bronze/reviews.parquet")

## Separate corrupt rows

Rows that failed to parse in bronze (`_corrupt_record` set) are split off from the good ones. There are none in this dataset, but the check stays in so a bad future load can't slip through.

In [3]:
bad = bronze.filter(F.col("_corrupt_record").isNotNull())
good = bronze.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
print("Corrupt: ", bad.count(), " Good: ", good.count())

Corrupt:  0  Good:  105698


## Basic cleaning

Trim whitespace and drop empty reviews, then remove exact duplicate reviews (105,698 → 104,028). Duplicates matter here: the same text landing in both train and test would inflate the scores.

In [4]:
clean = (good.withColumn("review", F.trim("review"))
         .filter(F.col("review").isNotNull() & (F.length("review") > 0)))
print("After removing nulls and triming reviews: ", clean.count())

After removing nulls and triming reviews:  105698


In [5]:
dedup = clean.dropDuplicates(["review"])
print("After removing dupes: ", dedup.count())

After removing dupes:  104028


## Sentiment labels

Binary label from the 1-5 rating:

- **1** (positive): rating 4 or 5
- **0** (negative): rating 1 or 2
- rating 3 (neutral) is dropped

In [6]:
labeled = (dedup.filter(F.col("rating") !=3)
           .withColumn("label", F.when(F.col("rating") >= 4, 1).otherwise(0)))
print("After labeling: ", labeled.select("review_id", "rating", "review", "label").show())

+---------+------+--------------------+-----+
|review_id|rating|              review|label|
+---------+------+--------------------+-----+
|      544|     5|“فندق ممتاز جدا”....|    1|
|     2606|     5|“فندق عائلي من ال...|    1|
|     2935|     4|“كانت اقامة ممتعة...|    1|
|     3090|     1|ضعيف جداً. لا شيئ...|    0|
|     3272|     5|استثنائي. . قلت ا...|    1|
|     5630|     4|جيد جداً. . اثناء...|    1|
|     6477|     4|جيد. . بطئ في الخ...|    1|
|     7333|     2|“اكثررررر من سيئ”...|    0|
|     7807|     1|“سيء جدا جدا”. لم...|    0|
|     8277|     4|“فندق مثالي بالمو...|    1|
|     8749|     2|“سي جدا سوء المعا...|    0|
|    11018|     5|استثنائي. . عدم و...|    1|
|    11166|     5|“تجربة مفرحة”. Lo...|    1|
|    12132|     5|“شكراً فندق القصي...|    1|
|    12445|     2|“إقامة لا بأس بها...|    0|
|    12831|     2|مخيب للأمل. الافط...|    0|
|    12863|     2|مخيب للأمل. . الخ...|    0|
|    15469|     5|“فندق اكثر من رائ...|    1|
|    16344|     1|ضعيف جداً. لم يع

## Check for label leakage

Many reviews start with a short title before the first `.`. Counting those titles per label shows that the most common ones are fixed words tied to the score, e.g. `مخيب للأمل` (disappointing) is always negative and `استثنائي` (exceptional) always positive.

A model would learn to read the title instead of the review, so these need to go.

In [7]:
first_part = F.split(F.regexp_replace("review", r'^[\"“”]+', ""), "\\.").getItem(0)

(labeled
    .withColumn("title", F.trim(first_part))
    .groupBy("title", "label")
    .count()
    .orderBy(F.desc("count"))
    .show(30, truncate=False))

+-----------+-----+-----+
|title      |label|count|
+-----------+-----+-----+
|مخيب للأمل |0    |9352 |
|جيد        |1    |7326 |
|استثنائي   |1    |6707 |
|ضعيف       |0    |6213 |
|ضعيف جداً  |0    |4873 |
|ممتاز      |1    |2714 |
|رائع       |1    |2519 |
|جيد جداً   |1    |2517 |
|ممتاز”     |1    |1067 |
|رائع”      |1    |602  |
|جيد”       |1    |574  |
|جيد جدا”   |1    |357  |
|مقبول”     |0    |323  |
|فندق رائع” |1    |296  |
|جميل”      |1    |283  |
|سيء”       |0    |243  |
|لا انصح به”|0    |239  |
|سيئ”       |0    |225  |
|فندق ممتاز”|1    |200  |
|ضعيف”      |0    |181  |
|فندق جميل” |1    |177  |
|سيء جدا”   |0    |145  |
|جيد”       |0    |142  |
|ممتاز جدا” |1    |124  |
|تجربة سيئة”|0    |122  |
|سيئ جدا”   |0    |107  |
|جيد جداً”  |1    |102  |
|فندق جيد”  |1    |93   |
|جميل جدا”  |1    |91   |
|فندق سيء”  |0    |90   |
+-----------+-----+-----+
only showing top 30 rows


## Remove the leaking titles

Strip the eight score-word titles from the start of the review into a new `review_clean` column (the original `review` is kept). Invisible Unicode characters (direction marks, zero-width spaces, tatweel `ـ`) are removed first, otherwise the pattern misses titles that look identical.

`had_title` records which rows were changed: 42,102 of them. The leftover check shows the remaining titles are in quotes (`“...”`), meaning the reviewer wrote them, so they are kept as part of the review.

In [8]:
titles = ["مخيب للأمل", "ضعيف جداً", "ضعيف", "جيد جداً", "جيد", "استثنائي", "ممتاز", "رائع"]

INVISIBLE = r"[\u200b-\u200f\u202a-\u202e\u061c\ufeff\u0640]"
DOT = r"[.\u06d4،]"
pattern = r"^\s*(" + "|".join(titles) + r")\s*" + DOT

no_leak = (labeled
    .withColumn("review_clean", F.regexp_replace("review", INVISIBLE, ""))
    .withColumn("had_title", F.col("review_clean").rlike(pattern))
    .withColumn("review_clean", F.trim(F.regexp_replace("review_clean", pattern, "")))
)

print("titles removed:", no_leak.filter(F.col("had_title")).count())

first_part = F.split(F.regexp_replace("review_clean", r'^[\"“”]+', ""), "\\.").getItem(0)
leftover = (no_leak.withColumn("title", F.trim(first_part))
            .groupBy("title").count().orderBy(F.desc("count")).limit(5).collect())
for row in leftover:
    print(row["count"], row["title"], [hex(ord(c)) for c in row["title"]])

titles removed: 42102
12039  []
1116 ممتاز” ['0x645', '0x645', '0x62a', '0x627', '0x632', '0x201d']
716 جيد” ['0x62c', '0x64a', '0x62f', '0x201d']
700 الموقع ['0x627', '0x644', '0x645', '0x648', '0x642', '0x639']
630 رائع” ['0x631', '0x627', '0x626', '0x639', '0x201d']


## Normalise the text

Applied to `review_clean`:

1. Remove quote marks.
2. Remove diacritics (tashkeel, code points U+064B to U+0652).
3. Remove emojis.
4. Collapse a character repeated 3+ times to one, e.g. `اكثررررر` → `اكثر`.
5. Collapse whitespace.

Reviews shorter than 3 characters are dropped, and duplicates are removed again since cleaning makes new ones (→ 103,255).

In [9]:
silver = (no_leak
    .withColumn("review_clean", F.regexp_replace("review_clean", r'[\"“”]', ""))
    .withColumn("review_clean", F.regexp_replace("review_clean", r'[\u064B-\u0652]', ""))
    .withColumn("review_clean", F.regexp_replace("review_clean", r'[\x{1F300}-\x{1FAFF}\x{2600}-\x{27BF}]', ""))
    .withColumn("review_clean", F.regexp_replace("review_clean", r'(.)\1{2,}', "$1"))
    .withColumn("review_clean", F.trim(F.regexp_replace("review_clean", r'\s+', " ")))
    .filter(F.length("review_clean") >= 3)
    .dropDuplicates(["review_clean"]) 
)
print("after normalization + dedup:", silver.count())

after normalization + dedup: 103255


## Split and write silver

Random 80 / 10 / 10 train / val / test split with a fixed seed (42) so it is reproducible. Labels come out close to 50/50 in every split.

Written as Parquet partitioned by `split`, so each set can be loaded on its own.

In [10]:
silver = (silver
    .withColumn("r", F.rand(seed=42))
    .withColumn("split", F.when(F.col("r") < 0.8, "train")
                          .when(F.col("r") < 0.9, "val")
                          .otherwise("test"))
    .select("review_id", "hotel_name", "user_type", "room_type", "nights",
            "rating", "label", "had_title", "review", "review_clean", "split")
)

silver.groupBy("split", "label").count().orderBy("split", "label").show()

(silver.write
    .mode("overwrite")
    .partitionBy("split")
    .parquet("../data/silver/hard_reviews"))

+-----+-----+-----+
|split|label|count|
+-----+-----+-----+
| test|    0| 5092|
| test|    1| 5143|
|train|    0|41648|
|train|    1|41113|
|  val|    0| 5142|
|  val|    1| 5117|
+-----+-----+-----+

